# Un primer clasificador nativo

[`06a`](06a_entrenamiento_variacional.ipynb) y [`06b`](06b_optimizadores_pso_qng.ipynb) entrenaron los parámetros libres de un circuito con `train()` y varios optimizadores. Este notebook aplica la misma idea a datos: un clasificador que aprende a distinguir dos grupos de puntos. En vez de un circuito escrito a mano, el modelo se monta con tres piezas de `polypus.qml`, y Polypus construye el circuito a partir de ellas. Como Polypus conoce ese circuito, puede calcular el gradiente exacto del coste, predecir con el modelo entrenado y guardarlo en un fichero.

## Los datos

Cada muestra es un punto con dos características, dos números entre 0 y π. Una clase se agrupa en torno a 0.8 y la otra en torno a 2.3, con algo de dispersión aleatoria sacada de una semilla fija, así que cada ejecución obtiene los mismos puntos. Las etiquetas son `+1` y `−1`, la convención de la pérdida que se usa más abajo.

`qml.Dataset` mantiene juntas las muestras y sus etiquetas y las comprueba una vez: todas las filas con la misma anchura, todos los valores finitos. `train_test_split` aparta una cuarta parte de las muestras, barajadas con su propia semilla, como conjunto de prueba que el modelo no ve durante el entrenamiento:

In [ ]:
import numpy as np
import polypus
from polypus import qml

rng = np.random.default_rng(3)
X = np.vstack([rng.normal(0.8, 0.3, (15, 2)), rng.normal(2.3, 0.3, (15, 2))]).clip(
    0, np.pi
)
y = np.repeat([1, -1], 15)

datos = qml.Dataset(X, y)
entrenamiento, prueba = datos.train_test_split(0.25, seed=7)
print(
    entrenamiento.num_samples,
    "muestras de entrenamiento,",
    prueba.num_samples,
    "de prueba",
)
print(entrenamiento.num_features, "características por muestra")

## Las tres piezas de un modelo

El **`Encoder`** carga cada muestra en los qubits. `angle_encoder("ry")` convierte la característica `j` en una rotación `ry` del qubit `j` con ese ángulo, así que dos características necesitan dos qubits. Por eso las características están entre 0 y π: un ángulo de 0 deja el qubit en `0`, y un ángulo de π lo voltea a `1`. El encoder no tiene parámetros entrenables, solo depende de los datos:

In [ ]:
encoder = qml.Encoder(2).angle_encoder("ry")
print(encoder.num_qubits, "qubits,", encoder.num_layers, "capa")

El **`Ansatz`** contiene las capas entrenables, cuyos ángulos ajusta el entrenamiento igual que los ángulos `Param` de [`06a`](06a_entrenamiento_variacional.ipynb). `real_amplitudes(1)` es una disposición estándar: una rotación `ry` en cada qubit, una `cx` entre ellos y otra `ry` en cada qubit, cuatro ángulos libres en total:

In [ ]:
ansatz = qml.Ansatz(2).real_amplitudes(1)
print(ansatz.num_params, "parámetros entrenables")

El **`Readout`** convierte la medida en una predicción. `qml.Z(0)` es el observable `Z` sobre el qubit 0: su valor esperado, escrito `⟨Z₀⟩`, va de `+1` cuando el qubit 0 se mide siempre como `0` a `−1` cuando es siempre `1`. Con `decision="sign"` la predicción es `+1` cuando `⟨Z₀⟩ ≥ 0` y `−1` en otro caso, una de las dos etiquetas:

In [ ]:
readout = qml.Readout([qml.Z(0)], decision="sign")
print("decisión", readout.decision, "sobre", readout.num_observables, "observable")

`qml.Model` junta las tres piezas. Comprueba que encajan, con el mismo número de qubits en el encoder y en el ansatz y un readout sobre esos qubits, y cuenta los parámetros entrenables:

In [ ]:
modelo = qml.Model(encoder, ansatz, readout)
print(modelo.num_params, "parámetros entrenables")

## Entrenar con `fit`

`model.fit` entrena el modelo con los mismos optimizadores que `train()`. Aquí es Adam, que sigue el gradiente del coste. En un modelo nativo Polypus calcula ese gradiente de forma exacta con la regla de desplazamiento de parámetros, parameter-shift: dos circuitos más por cada puerta entrenable, con su ángulo desplazado hacia arriba y hacia abajo, en vez de una aproximación.

Con etiquetas, `fit` necesita una **pérdida**, un número que dice lo lejos que está cada predicción de su etiqueta. `loss="hinge"` está pensada para etiquetas `±1`: puntúa cada muestra como `1 − y·⟨Z₀⟩`, que vale 0 cuando `⟨Z₀⟩` es igual a la etiqueta `y`, 1 cuando `⟨Z₀⟩` es 0 y 2 cuando es igual a la etiqueta contraria. Igual que en [`06a`](06a_entrenamiento_variacional.ipynb), el optimizador maximiza un fitness, aquí la pérdida media con el signo cambiado, así que el mejor valor posible es 0.

`exact=True` calcula las probabilidades de cada circuito de forma exacta en vez de muestrear shots, así que el fitness no lleva ruido de muestreo; [`07b`](07b_lecturas_y_perdidas.ipynb) lo compara con entrenar con shots. La `seed` fija el punto de partida, así que la ejecución se repite exactamente igual. `entrenamiento` ya lleva sus etiquetas, así que no se pasan otra vez:

In [ ]:
result = modelo.fit(
    entrenamiento,
    method=polypus.Adam(max_iters=60, learning_rate=0.1),
    loss="hinge",
    exact=True,
    seed=7,
    id="primer_clasificador",
)
print(result.best_fitness)

## Los campos de `TrainResult`

`fit` devuelve el mismo `TrainResult` que `train()` en [`06a`](06a_entrenamiento_variacional.ipynb). `best_params` y `best_fitness` son el mejor punto encontrado y su fitness, `iterations_run` cuenta los pasos de Adam que se han ejecutado de verdad de los 60 que permite `max_iters`, y `converged` dice si paró antes porque el fitness había dejado de mejorar. `fitness_history` guarda el mejor fitness tras cada iteración, así que su curva muestra cómo ha ido el entrenamiento:

In [ ]:
import matplotlib.pyplot as plt

print(len(result.best_params), "parámetros:", result.best_params)
print("iteraciones:", result.iterations_run, "| convergido:", result.converged)
print(len(result.fitness_history), "valores en fitness_history")

plt.figure(figsize=(5, 3))
plt.plot(range(1, len(result.fitness_history) + 1), result.fitness_history)
plt.xlabel("iteración")
plt.ylabel("mejor fitness")
plt.show()

## Predecir con `bind` y `predict`

`best_params` no son más que números. `model.bind` los fija en el modelo y devuelve un `TrainedModel`, listo para inferencia, mientras que el `Model` no cambia. `predict` devuelve una decisión por fila, aquí `+1` o `−1`, y acepta filas o un `Dataset` entero. Igual que `fit`, admite `exact=True`:

In [ ]:
entrenado = modelo.bind(result.best_params)
predicciones = entrenado.predict(prueba, exact=True)
print("predicciones:", predicciones)
print("etiquetas:   ", [int(etiqueta) for etiqueta in prueba.labels])

## Exactitud en el conjunto de prueba

La exactitud es la fracción de muestras de prueba cuya predicción coincide con su etiqueta. Estas muestras no han participado en el entrenamiento, así que la exactitud mide cómo se comporta el modelo con puntos que no ha visto:

In [ ]:
aciertos = sum(p == etiqueta for p, etiqueta in zip(predicciones, prueba.labels))
print("exactitud en prueba:", aciertos / prueba.num_samples)

## Guardar y cargar

`save` escribe el modelo, sus capas y sus parámetros entrenados en un fichero JSON, y `qml.load` lo lee de nuevo como un `TrainedModel`. Aquí el fichero va a una carpeta temporal que desaparece al terminar la celda. Los parámetros se leen exactamente iguales, así que el modelo cargado predice lo mismo que el original:

In [ ]:
import os
import tempfile

with tempfile.TemporaryDirectory() as carpeta:
    ruta = os.path.join(carpeta, "primer_clasificador.json")
    entrenado.save(ruta)
    cargado = qml.load(ruta)

print("mismos parámetros:", cargado.params == entrenado.params)
print("mismas predicciones:", cargado.predict(prueba, exact=True) == predicciones)

## Resumen

Este notebook ha construido un primer clasificador nativo: un `Dataset` dividido en muestras de entrenamiento y de prueba, un modelo hecho de un `Encoder`, un `Ansatz` y un `Readout`, entrenado con `fit` y Adam sobre el gradiente exacto de la pérdida hinge, un `TrainResult` con su `fitness_history`, predicciones con `bind` y `predict`, la exactitud sobre muestras que el modelo no había visto, y un modelo guardado en un fichero y cargado de nuevo.

El siguiente notebook cambia la lectura y la pérdida para resolver otros tipos de problemas.

## Siguiente paso

Continuar con: [`07b_lecturas_y_perdidas.ipynb`](07b_lecturas_y_perdidas.ipynb).